In [1]:
%cd ../../../

/home/hoanghu/projects/Food-Waste-Optimization


In [2]:
import re
import sys
from pathlib import Path
from itertools import zip_longest, combinations

import networkx as nx

import numpy as np
import pandas as pd
from loguru import logger


In [3]:
logger.remove() #remove the old handler. Else, the old one will work along with the new one you've added below'
logger.add(sys.stdout, level="DEBUG")

1

# Add meals

In [4]:
def hamming_distance(s1: str, s2: str) -> float:
    min_len = min(len(s1), len(s2))
    dist = sum(c1 != c2 for c1, c2 in zip_longest(s1, s2))  * 1.0 / min_len

    return dist

In [5]:
meals = {}

## Process meal list

In [6]:
meals_raw = pd.read_excel("data/processed/phase_4/menus.xlsx", sheet_name="meals")

meals_raw.head()

,meal_code,meal_name,category,CO2
0,34.0,Sitruunaiset kalapaloja ja kukkakaalitsatsikia,kala,0.81
1,710.0,Broileri-Caesarsalaatti,kana,0.67
2,724.0,Broilerilasagnette,kana,0.82
3,725.0,"Broilerinuggetit, currykastiketta",kana,1.06
4,726.0,"Broileripyörykät, currykastike",kana,0.86


In [7]:
# Remove duplicate entries
meal_list = (
    meals_raw
    .dropna(axis=0, how='any')
    .drop(columns='CO2')
    .groupby('meal_code')
    .last()
    .reset_index()
    .rename(columns={
        'meal_code': 'meal_id',
        'meal_name': 'meal',
        'category': 'meal_type',
    })
)


# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({'meal': meal_name, 'tag': tag})
    
meal_list['meal'] = meal_list['meal'].str.strip().apply(_f_extract_tag)['meal']



meal_list['meal_type'] = meal_list['meal_type'].str.strip().map({
    'kala': 'fish',
    'liha': 'meat',
    'kana': 'chicken',
    'vegaani': 'vegan',
    'kasvis': 'vegetarian',
    'keskiarvo': 'buffet',
})

meal_list['meal_id'] = meal_list['meal_id'].astype(int)

meal_list = meal_list.sort_values('meal_id').groupby('meal').first().reset_index()

meal_list = meal_list[~meal_list['meal'].str.lower().str.contains('take away')]

meal_list['schoolyear'] = "23-24"


meal_list.head()

,meal,meal_id,meal_type,schoolyear
0,"""Butter"" härkäpapua & pähkinää",9017,vegan,23-24
1,2023 Härkäpu-sienilasagnette,7201,vegan,23-24
2,Appelisiini-luomukikhernecurrya,9032,vegan,23-24
3,Artisokkavugetteja & tuoretomaattisalsaa,9102,vegan,23-24
4,Aurajuusto-pinaattilasagnette,7010,vegetarian,23-24


In [8]:
THRES = 0.1

for meal_other in meal_list.itertuples():
    if meal_other.meal in meals:    # exact match
        # With meal_list, we dont care exact match case
        logger.debug("herer 1")
    else:
        # Find approximate match
        best_meal = None
        best_dist = 1e10
        for meal in meals.keys():
            dist = hamming_distance(meal_other.meal, meal)

            if dist <= THRES and dist < best_dist:
                best_dist = dist
                best_meal = meal

        # Handle 2 cases: found approximate match and not found
        if best_meal is None:   # not found
            meals[meal_other.meal] = {
                'meal_id': meal_other.meal_id,
                'meal_type_1': meal_other.meal_type,
                'schoolyear': meal_other.schoolyear,
                'is_kela': False,
                'is_new': False,
                'restaurant': None,
                'meal_type_2': None,
                'aliases': []
            }
        else:                   # found approximate match
            logger.debug(f"herer 2: {meal_other.meal} - {best_meal}")

            meals[best_meal]['aliases'].append(meal_other.meal)

2024-11-06 14:37:52.715 | DEBUG    | __main__:<module>:31 - herer 2: Lounaspatonki, savu-nyhtökaura - Lounaspatonki, Savu-Nyhtökaura
2024-11-06 14:37:52.778 | DEBUG    | __main__:<module>:31 - herer 2: Mustajuurikeittoa - Mustajuurikeitto
2024-11-06 14:37:52.985 | DEBUG    | __main__:<module>:31 - herer 2: Rapeat kasvisjauhispihvit, mangokastike - Rapeat kasvisjauhispihvit, Mangokastike


# Merge meals in menu to `meals`

## Process menu file

In [9]:
path = "data/processed/phase_4/menus.xlsx"

weeks = ["week1", "week2", "week3", "week4", "week5", "week6"]

list_menus = []
for week in weeks:
    raw = pd.read_excel(path, sheet_name=week)
    raw['week'] = week

    list_menus.append(raw)

menus_raw = pd.concat(list_menus)
menus_raw.head()

,meal_type,misc,monday,tuesday,wednesday,thursday,friday,saturday,week
0,today’s special,meal_id,7609,7607,9039,6121,6818,2284,week1
1,today’s special,meal_name,"Lohta pesto & mustajuurta (L, G)","Broileria pekonikastikkeessa (L, G, KELA)",BBQ-savutofuburger & raikasta nektariiniketsup...,"Karamellisoitua possua (M,G,KELA)",Filippiiniläiset kanavartaat & Hedelmäsalsaa (...,"Limemarinoidut kanavartaat, avokadokastiketta ...",week1
2,vegan-kpl,meal_id,9044,9064,9053,9097,9052,9075,week1
3,vegan-kpl,meal_name,Kasvisjalapenonugetteja ja Chipotle-majoneesia...,"Kasvispyöryköitä ja Currykastiketta (VE, G, KE...","Falafelpyöryköitä & Chimicurrikastiketta (VE, ...",Kasvisjahispyöryköitä pesto-tomaattikastikees...,Punajuuripyöryköitä ja vaaleaa balsamicokastik...,"Pinaattilettuja & puolukkasurvosta (VE, G, IV)",week1
4,vegan-miscellaneous,meal_id,9018,7562,7573,6852,7564,9073,week1


In [10]:
meal_ids_new = [
    9039,
    9044, 9064, 9053, 9097, 9052, 9075,
    9018, 9073,
    9074,
    9078,
    9105, 9106,
    9107, 8994,

    9026, 9022,
    9049, 9059, 6044, 8985, 9056, 9066,
    9042,
    9079,
    2218, 2206, 20018,
    8986,

    9094,
    9045, 9063, 9047, 9069, 9054, 9067,
    8993, 9017, 9036,
    9076,
    8989,
    200011, 20003,

    9029,
    9060, 9051, 8983, 9046, 7029, 9098,
    8999,
    8988,
    20017, 200011, 950005,

    9050, 9068, 9062, 9048, 9065,
    9003, 9111,
    9077, 9037,
    8987, 8992,
    950017, 950019, 950000,

    9021, 9040,
    9096, 9061, 9055, 9043, 9058,
    9099,
    8991,
    950001, 950011,
]

cols = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']

menus = (
    menus_raw
    .melt(id_vars=['meal_type', 'misc', 'week'], value_vars=cols, value_name='meal', var_name='weekday')
    .pivot(index=['meal_type', 'week', 'weekday'], columns='misc', values='meal')
    .reset_index()
    .dropna(axis=0, how='any')
)
# menus['meal_id'] = menus['meal_id'].astype(np.int32)

# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({'meal': meal_name, 'tag': tag})
    
extracted = menus['meal_name'].apply(_f_extract_tag)
menus = pd.concat([menus.drop(columns='meal_name'), extracted], axis=1)

# Extract `is_kela`
menus['is_kela'] = menus['tag'].str.contains('KELA')

# Mark new dishes
menus['is_new'] = menus['meal_id'].isin(meal_ids_new)

# Process meal_id
pat_meal_code = r"(\d*)\s*\/\s*(\d*)"

def _f_extract_meal_id(s):
    meal_id_1, meal_id_2 = None, None
    match s:
        case int():
            meal_id_1 = s
        case str():
            out = re.findall(pat_meal_code, s)[0]
            meal_id_1, meal_id_2 = int(out[0]), int(out[1])

    return pd.Series({'meal_id_1': meal_id_1, 'meal_id_2': meal_id_2})

extracted = menus['meal_id'].apply(_f_extract_meal_id)
menus = pd.concat([menus, extracted], axis=1)

# Remove duplicate meals
menus = menus.groupby('meal_id').first().reset_index()

# Assign restaurant
menus['restaurant'] = None

meal_type_che_exac = ['fish', 'meat', 'today’s special', 'vegan-kpl', 'vegan-miscellaneous']
indices = menus[menus['meal_type'].isin(meal_type_che_exac)].index
menus.loc[indices, 'restaurant'] = 'che-exa'

meal_type_phy = 'salad'
indices = menus[menus['meal_type'] == meal_type_phy].index
menus.loc[indices, 'restaurant'] = 'phy'


def _extract_meal_type(s: str):
    meal_type_1, meal_type_2 = None, None

    match s:
        case 'meat' | 'fish':
            meal_type_1 = s
        case _:
            meal_type_2 = s

    return pd.Series({'meal_type_1': meal_type_1, 'meal_type_2': meal_type_2})

tmp = menus['meal_type'].str.strip().apply(_extract_meal_type)
menus['meal_type_1'] = tmp['meal_type_1'].copy()
menus['meal_type_2'] = tmp['meal_type_2'].copy()


menus.drop(columns=['week', 'weekday', 'tag', 'meal_id', 'meal_type'], inplace=True)
menus.rename(columns={'meal_id_1': 'meal_id'}, inplace=True)

menus.head()

,meal,is_kela,is_new,meal_id,meal_id_2,restaurant,meal_type_1,meal_type_2
0,Broileri-Caesarsalaatti,True,False,710.0,NaN,phy,None,salad
1,Meksikolainen uunimakkara,True,False,785.0,NaN,che-exa,meat,None
2,Uunimakkara ja sinappikastike,True,False,790.0,NaN,che-exa,meat,None
3,Carbonara-kastike & pastaa,False,False,791.0,NaN,che-exa,meat,None
4,Chili-katkarapusalaatti,True,False,839.0,NaN,phy,None,salad


### Compose list of baguettes

In [11]:
baguettes_raw = pd.read_excel("data/processed/phase_4/menus.xlsx", sheet_name="baguettes")

baguettes_raw.head()

,meal_id,meal,meal_type,co2,is_kela
0,1445,"Lounaspatonki, juusto",kasvis,0.79,NaN
1,1446,"Lounaspatonki, kalkkunatahna",kana,0.53,NaN
2,1447,"Lounaspatonki, kasvis",vegaani,0.35,KELA
3,1448,"Lounaspatonki, katkarapu",kala,1.04,KELA
4,1449,"Lounaspatonki, kinkku",liha,0.46,NaN


In [12]:
baguettes = baguettes_raw.copy()


# Remove unnecessary columns
baguettes.drop(columns='co2', inplace=True)

# Rename meal_type
baguettes['meal_type'] = baguettes['meal_type'].map({
    'kasvis': 'vegetarian',
    'kana': 'chicken',
    'vegaani': 'vegan',
    'kala': 'fish',
    'not mapped': np.nan,
    'liha': 'meat'
})
baguettes.rename(columns={'meal_type': 'meal_type_1'}, inplace=True)


# Add other columns
baguettes['restaurant'] = 'phy'
baguettes['is_new'] = False
baguettes['is_kela'] = baguettes['is_kela'].apply(lambda x: x is not np.nan)
baguettes['meal_type_2'] = 'baguettes'

baguettes.head()

,meal_id,meal,meal_type_1,is_kela,restaurant,is_new,meal_type_2
0,1445,"Lounaspatonki, juusto",vegetarian,False,phy,False,baguettes
1,1446,"Lounaspatonki, kalkkunatahna",chicken,False,phy,False,baguettes
2,1447,"Lounaspatonki, kasvis",vegan,True,phy,False,baguettes
3,1448,"Lounaspatonki, katkarapu",fish,True,phy,False,baguettes
4,1449,"Lounaspatonki, kinkku",meat,False,phy,False,baguettes


In [13]:
menus = pd.concat([menus, baguettes], ignore_index=True)
menus['schoolyear'] = "24-25"


menus.head()

,meal,is_kela,is_new,meal_id,meal_id_2,restaurant,meal_type_1,meal_type_2,schoolyear
0,Broileri-Caesarsalaatti,True,False,710.0,NaN,phy,None,salad,24-25
1,Meksikolainen uunimakkara,True,False,785.0,NaN,che-exa,meat,None,24-25
2,Uunimakkara ja sinappikastike,True,False,790.0,NaN,che-exa,meat,None,24-25
3,Carbonara-kastike & pastaa,False,False,791.0,NaN,che-exa,meat,None,24-25
4,Chili-katkarapusalaatti,True,False,839.0,NaN,phy,None,salad,24-25


## Merge `meals` with 2024-2025 menu

In [14]:
THRES = 0.1

for meal_other in menus.itertuples():
    # Find exact match via name
    if meal_other.meal in meals:
        # logger.debug(f"Found exact match via name: {meal_other.meal}")

        meal = meals[meal_other.meal]

        meal['meal_type_1'] = meal['meal_type_1'] or meal_other.meal_type_1
        meal['schoolyear'] = max(meal['schoolyear'], meal_other.schoolyear)
        meal['is_kela'] = meal_other.is_kela
        meal['is_new'] = meal_other.is_kela
        meal['restaurant'] = meal_other.restaurant
        meal['meal_type_2'] = meal_other.meal_type_2

        continue
    
    # Find exact match via meal_id
    found = False
    for info in meals.values():
        if meal_other.meal_id == info['meal_id']:   # Found match meal_id
            if meal_other.meal_id == 790:
                logger.debug(f"Found exact match via meal_id: {meal_other.meal_id}")

            found = True

            info['aliases'].append(meal_other.meal)
            info['meal_type_1'] = info['meal_type_1'] or meal_other.meal_type_1
            info['schoolyear'] = info['schoolyear'] or meal_other.schoolyear
            info['is_kela'] = meal_other.is_kela
            info['is_new'] = meal_other.is_kela
            info['restaurant'] = meal_other.restaurant
            info['meal_type_2'] = meal_other.meal_type_2

            break
    if found is True:
        continue

    # Find approximate match
    best_meal = None
    best_dist = 1e10
    
    for meal in meals.keys():
        dist = hamming_distance(meal_other.meal, meal)
        
        if dist <= THRES and dist < best_dist:
            best_dist = dist
            best_meal = meal

    # Handle 2 cases: found approximate match and not found
    if best_meal is None:           # not found
        meals[meal_other.meal] = {
            'meal_id': meal_other.meal_id,
            'meal_type_1': meal_other.meal_type_1,
            'schoolyear': meal_other.schoolyear,
            'is_kela': meal_other.is_kela,
            'is_new': meal_other.is_new,
            'restaurant': meal_other.restaurant,
            'meal_type_2': meal_other.meal_type_2,
            'aliases': []
        }
    else:                           # found approximate match
        logger.debug(f"Found approximate match: {meal_other.meal} - {best_meal}")

        info = meals[best_meal]

        info['aliases'].append(meal_other.meal)
        info['schoolyear'] = max(info['schoolyear'], meal_other.schoolyear)
        info['is_kela'] = info['is_kela'] or meal_other.is_kela
        info['is_new'] = info['is_new'] or meal_other.is_new
        info['restaurant'] = info['restaurant'] or meal_other.restaurant
        info['meal_type_1'] = info['meal_type_1'] or meal_other.meal_type_1
        info['meal_type_2'] = meal_other.meal_type_2

2024-11-06 14:37:54.249 | DEBUG    | __main__:<module>:24 - Found exact match via meal_id: 790.0
2024-11-06 14:37:54.264 | DEBUG    | __main__:<module>:64 - Found approximate match: TexMex-siemenpyöryköitä ja Louisana-kastiketta - TexMex-siemenpyöryköitä ja Louisana-kastikett
2024-11-06 14:37:54.280 | DEBUG    | __main__:<module>:64 - Found approximate match: Marokkolaiset kasvispihvit & Chermoula kastiketta - Marokkolaiset kasvispihvit & Chermoula kastik
2024-11-06 14:37:54.297 | DEBUG    | __main__:<module>:64 - Found approximate match: Täysjyväkalafileetä & lime-korianteriremouladea - Täysjyväkalafileetä & lime-korianteriremoulad


# Merge with meals from POS

## Process POS files

In [15]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
]

raw = []
for path in paths:
    df = pd.read_csv(path, delimiter=';')
    df.columns = np.arange(df.shape[1])
    raw.append(df)


pos_raw = pd.concat(raw, ignore_index=True)
pos_raw.head()

/tmp/ipykernel_59087/1943815052.py:9: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, delimiter=';')


,0,1,2,3,4,5,6
0,2.1.2023,10:31,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
1,2.1.2023,10:32,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
2,2.1.2023,10:32,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
3,2.1.2023,10:35,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
4,2.1.2023,10:36,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",2,"1,8"


In [16]:
pos = pos_raw.copy()


# Rename columns
pos.columns = np.arange(pos.shape[1])
pos.rename(
    columns={
        0: 'date',
        1: 'time',
        2: 'restaurant',
        3: 'meal_type',
        4: 'meal',
        5: 'pcs',
        6: 'co2',
    },
    inplace=True
)


# Process date
pos['date'] = pd.to_datetime(pos['date'], format="%d.%m.%Y")



# Process meal_type
pos['meal_type'] = pos['meal_type'].map({
    'Liha': 'meat',
    'Kala': 'fish',
    'Vegaani': 'vegan',
    'Kasvis': 'vegetarian',
    'Kana': 'chicken'
})


# Process meal
pos['meal'] = pos['meal'].str.strip()


# Map restaurant name
pos['restaurant'] = pos['restaurant'].map({
    '600 Chemicum': 'che-exa', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'che-exa', #'exactum'
    '570 Viikuna': 'vik',
})


# Get pair of meal and meal_type
meals_pos = pos.groupby(['meal', 'meal_type']).last().reset_index()


# Extract schoolyear
meals_pos['schoolyear'] = meals_pos['date'].apply(lambda x: '24-25' if x >= pd.to_datetime("2024-09-01") else '23-24')


# Keep necessary columns
meals_pos = meals_pos[['meal', 'meal_type', 'restaurant', 'schoolyear']]


# Remove take away
meals_pos = meals_pos[~meals_pos['meal'].str.lower().str.contains('take away')]



meals_pos.head()

,meal,meal_type,restaurant,schoolyear
0,"""Butter"" luomukikhernekastiketta",vegan,che-exa,24-25
1,Aurajuusto-pinaattilasagnettea,vegetarian,che-exa,24-25
2,BBQ-Broilerikastiketta,chicken,che-exa,24-25
3,Bangladeshilainen linssipata,vegan,che-exa,23-24
4,Bataatti-maapähkinäkeitto,vegan,che-exa,24-25


## Merge meals from POS data with `meals`

In [17]:
THRES = 0.1

for meal_other in meals_pos.itertuples():
    # Find exact match via name
    if meal_other.meal in meals:
        # logger.debug(f"Found exact match via name: {meal_other.meal}")

        meal = meals[meal_other.meal]

        meal['meal_type_1'] = meal['meal_type_1'] or meal_other.meal_type_1
        meal['schoolyear'] = max(meal['schoolyear'], meal_other.schoolyear)
        meal['restaurant'] = meal['restaurant'] or meal_other.restaurant

        continue
    

    # Find approximate match
    best_meal = None
    best_dist = 1e10
    
    for meal in meals.keys():
        dist = hamming_distance(meal_other.meal, meal)
        
        if dist <= THRES and dist < best_dist:
            best_dist = dist
            best_meal = meal

    # Handle 2 cases: found approximate match and not found
    if best_meal is None:           # not found
        meals[meal_other.meal] = {
            'meal_id': -1,
            'meal_type_1': meal_other.meal_type,
            'schoolyear': meal_other.schoolyear,
            'is_kela': False,
            'is_new': False,
            'restaurant': meal_other.restaurant,
            'meal_type_2': None,
            'aliases': []
        }
    else:                           # found approximate match
        logger.debug(f"Found approximate match: {meal_other.meal} - {best_meal}")

        info = meals[best_meal]

        info['aliases'].append(meal_other.meal)
        info['schoolyear'] = max(info['schoolyear'], meal_other.schoolyear)
        info['restaurant'] = info['restaurant'] or meal_other.restaurant
        info['meal_type_1'] = info['meal_type_1'] or meal_other.meal_type

2024-11-06 14:37:55.448 | DEBUG    | __main__:<module>:41 - Found approximate match: Aurajuusto-pinaattilasagnettea - Aurajuusto-pinaattilasagnette


2024-11-06 14:37:55.464 | DEBUG    | __main__:<module>:41 - Found approximate match: BBQ-Broilerikastiketta - BBQ-broilerikastiketta
2024-11-06 14:37:55.501 | DEBUG    | __main__:<module>:41 - Found approximate match: Bataattipihvit,curry-minttusoi - Bataattipihvit,curry-minttukas
2024-11-06 14:37:55.536 | DEBUG    | __main__:<module>:41 - Found approximate match: Butter Chicken - Butter chicken
2024-11-06 14:37:55.541 | DEBUG    | __main__:<module>:41 - Found approximate match: Chili con Peas of Heaven - Chili con peas of heaven
2024-11-06 14:37:55.548 | DEBUG    | __main__:<module>:41 - Found approximate match: Chili con nyhtis - Chili con Nyhtis
2024-11-06 14:37:55.585 | DEBUG    | __main__:<module>:41 - Found approximate match: Herkkusienikeittoa - Herkkusienikeitto
2024-11-06 14:37:55.610 | DEBUG    | __main__:<module>:41 - Found approximate match: Jambalayaa & paahdettua halloumia - Jambalayaa & paahdettua Halloumia
2024-11-06 14:37:55.627 | DEBUG    | __main__:<module>:41 - Foun

# Finalize `meals`

In [18]:
meals = pd.DataFrame(meals).T

meals.head()

,meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,aliases
"""Butter"" härkäpapua & pähkinää",9017,vegan,24-25,True,True,che-exa,vegan-miscellaneous,[]
2023 Härkäpu-sienilasagnette,7201,vegan,23-24,False,False,None,None,[]
Appelisiini-luomukikhernecurrya,9032,vegan,23-24,False,False,None,None,[]
Artisokkavugetteja & tuoretomaattisalsaa,9102,vegan,23-24,False,False,None,None,[]
Aurajuusto-pinaattilasagnette,7010,vegetarian,24-25,False,False,che-exa,None,[Aurajuusto-pinaattilasagnettea]


## Post-process `meals`

In [19]:
meals = meals.reset_index()

# Rename column
meals.rename(columns={'index': 'meal'}, inplace=True)


# Assign meal_id
indices = meals[meals['meal_id'] == -1].index
max_idx = meals['meal_id'].max() + 1
meals.loc[indices, 'meal_id'] = np.arange(max_idx, max_idx + len(indices))


# Extract dim `meal_names`
meal_names = []
for meal in meals.itertuples():
    meal_names.append({'meal_id': meal.meal_id, 'meal': meal.meal})

    for alias in meal.aliases:
        meal_names.append({'meal_id': meal.meal_id, 'meal': alias})

meal_names = pd.DataFrame.from_records(meal_names)


# Remove unncessary columns
meals.drop(columns=['meal', 'aliases'], inplace=True)

### Configure list of panini

In [20]:
tmp = meal_names.copy()

tmp['panini'] = tmp['meal'].str.lower().str.contains('panini').apply(lambda x: 'panini' if x is True else None)
tmp = (
    tmp.groupby('meal_id')
    .first()
    .reset_index()
    .drop(columns='meal')
)


meals = (
    meals
    .merge(tmp, on='meal_id', how='left')
)
meals['meal_type_2'] = meals[['meal_type_2', 'panini']].bfill(axis=1).iloc[:, 0]
meals.drop(columns='panini', inplace=True)

meals.head()

,meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2
0,9017,vegan,24-25,True,True,che-exa,vegan-miscellaneous
1,7201,vegan,23-24,False,False,None,None
2,9032,vegan,23-24,False,False,None,None
3,9102,vegan,23-24,False,False,None,None
4,7010,vegetarian,24-25,False,False,che-exa,None


## Unit tests

In [21]:
t = meals.groupby('meal_id')['meal_type_1'].count()

assert len(t[t> 1]) == 0

In [22]:
meals[meals['meal_id'] == 790.0]

,meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2
710,790,meat,23-24,True,True,che-exa,None


In [23]:
tmp = (
    menus
    .merge(meal_names, on='meal', how='left')
)


assert len(tmp[tmp['meal_id_y'].isna()]) == 0

In [24]:
tmp = (
    meal_list
    .merge(meal_names, on='meal', how='left')
)


assert len(tmp[tmp['meal_id_y'].isna()]) == 0

In [25]:
tmp = (
    meals_pos
    .merge(meal_names, on='meal', how='left')
)


assert len(tmp[tmp['meal_id'].isna()]) == 0

# Save dim

In [26]:
meals.to_excel("dim_meals.xlsx", index=False)
meal_names.to_excel("dim_meal_names.xlsx", index=False)

## Check potentially identical meals

Oct 31: No pair of identical meals found!

In [25]:
# meal_names = meals_final['meal']

# def hamming_distance(s1, s2):
#     return sum(c1 != c2 for c1, c2 in zip_longest(s1, s2))

# sames = {meal: [] for meal in meal_names}

# for (s1, s2) in combinations(meal_names, 2):
#     min_len = min(len(s1), len(s2))
#     dist = hamming_distance(s1, s2) * 1.0 / min_len
#     if dist < .5:
#         sames[s1].append(s2)
#         sames[s2].append(s1)

# g = nx.from_dict_of_lists(sames)

# meals_groupped = []
# for x in nx.connected_components(g):
#     if len(x) > 1:
#         print(x)
    
#     meals_groupped.append(list(x))

# # nx.draw(g, with_labels = True)


# with open("meals_2024.json", "w+", encoding='utf-8') as f:
#     json.dump(meals_groupped, f, indent=2, ensure_ascii=False)